# 02C — ClinVar consequence deep analysis


In [17]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import fisher_exact, chi2_contingency, mannwhitneyu, kruskal, spearmanr, ks_2samp, entropy, norm
from IPython.display import display

PROCESSED = Path('../../data/processed')
df = pd.read_csv(PROCESSED / 'DMD_variants_annotated.csv')

print('Loaded:', PROCESSED / 'DMD_variants_annotated.csv')
print('Shape:', df.shape)


Loaded: ../../data/processed/DMD_variants_annotated.csv
Shape: (11308, 30)


In [18]:
import sys
from pathlib import Path

PROJECT_ROOT = None
for rel in ('../..', '..', '.'):
    cand = (Path.cwd() / rel).resolve()
    if (cand / 'src' / 'utils.py').exists():
        PROJECT_ROOT = cand
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('project root with src/utils.py not found')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import (
    add_consistency_flags,
    bh_adjust,
    chi2_table,
    ecdf_xy,
    fisher_bool,
    frame_mismatch,
    kruskal_group,
    mann_whitney_bool,
    mut_cons_mismatch,
    odds_ratio_ci,
    path_cons_mismatch,
    prepare_eda_dataframe,
    spearman_xy,
)

d = prepare_eda_dataframe(df)

print('Prepared shape:', d.shape)


Prepared shape: (11308, 59)


## 1. 📊 Consequence with pathogenic stacked


In [19]:
top_cons = d['consequence'].value_counts().head(15).index
tmp = d[d['consequence'].isin(top_cons)][['consequence', 'pathogenic']]
fig = px.histogram(tmp, x='consequence', color='pathogenic', barmode='stack', title='Consequence x pathogenic (top)')
fig.update_xaxes(tickangle=-45)
fig.show()


## 2. 🧪 χ² consequence ~ pathogenic


In [20]:
tmp = d[d['consequence'].isin(d['consequence'].value_counts().head(20).index)]
tab, chi2, p, dof = chi2_table(tmp, 'consequence', 'pathogenic')
display(tab)
display(pd.Series({'chi2': chi2, 'dof': dof, 'p_value': p}).to_frame('value'))


pathogenic,False,True
consequence,,
3 prime UTR variant,55,0
3 prime UTR variant|intron variant,14,2
5 prime UTR variant|intron variant,26,0
frameshift variant,1,595
frameshift variant|5 prime UTR variant,0,95
frameshift variant|intron variant,1,35
inframe_deletion,25,3
intron variant,2175,60
missense variant,2865,23


,value
chi2,8485.754153
dof,19.000000
p_value,0.000000


## 3. 📊 Consequence with phenotype


In [21]:
top_cons = d['consequence'].value_counts().head(12).index
tmp = d[d['consequence'].isin(top_cons)][['consequence', 'phenotype']]
fig = px.histogram(tmp, x='consequence', color='phenotype', barmode='stack', title='Consequence x phenotype (top)')
fig.update_xaxes(tickangle=-45)
fig.show()


## 4. 🧪 χ² consequence ~ phenotype


In [22]:
tmp = d[d['consequence'].isin(d['consequence'].value_counts().head(20).index)]
tab, chi2, p, dof = chi2_table(tmp, 'consequence', 'phenotype')
display(tab)
display(pd.Series({'chi2': chi2, 'dof': dof, 'p_value': p}).to_frame('value'))


phenotype,BMD,DMD
consequence,,
3 prime UTR variant,1,0
3 prime UTR variant|intron variant,1,9
5 prime UTR variant|intron variant,3,0
frameshift variant,42,419
frameshift variant|5 prime UTR variant,2,71
frameshift variant|intron variant,8,21
inframe_deletion,4,20
intron variant,229,1420
missense variant,363,2052


,value
chi2,2.123376e+02
dof,1.900000e+01
p_value,1.179522e-34


## 5. 📊 consequence with frame_status


In [23]:
top_cons = d['consequence'].value_counts().head(12).index
tmp = d[d['consequence'].isin(top_cons)][['consequence', 'frame']]
fig = px.histogram(tmp, x='consequence', color='frame', barmode='stack', title='Consequence x frame_status (top)')
fig.update_xaxes(tickangle=-45)
fig.show()


## 6. 🧪 χ² consequence ~ frame


In [24]:
tmp = d[d['consequence'].isin(d['consequence'].value_counts().head(20).index)]
tab, chi2, p, dof = chi2_table(tmp, 'consequence', 'frame')
display(tab)
display(pd.Series({'chi2': chi2, 'dof': dof, 'p_value': p}).to_frame('value'))


frame,in-frame,out-of-frame
consequence,,
3 prime UTR variant|intron variant,0,4
frameshift variant,0,596
frameshift variant|5 prime UTR variant,0,95
frameshift variant|intron variant,0,36
inframe_deletion,0,1
intron variant,4,369
missense variant,2687,201
missense variant|5 prime UTR variant,309,26
missense variant|intron variant,80,6


,value
chi2,4449.427882
dof,17.000000
p_value,0.000000


## 7. 📊 consequence vs exon density


In [25]:
top_cons = d['consequence'].value_counts().head(10).index
tmp = d[d['consequence'].isin(top_cons)][['consequence', 'exon_num']].dropna()
fig = px.violin(tmp, x='consequence', y='exon_num', box=True, points=False, title='Consequence vs exon distribution')
fig.update_xaxes(tickangle=-45)
fig.show()


## 8. 📊 consequence vs domain


In [26]:
top_cons = d['consequence'].value_counts().head(10).index
top_dom = d['domain_clean'].value_counts().head(12).index
tmp = d[d['consequence'].isin(top_cons) & d['domain_clean'].isin(top_dom)]
heat = pd.crosstab(tmp['consequence'], tmp['domain_clean'])
fig = px.imshow(heat, aspect='auto', color_continuous_scale='Blues', title='Consequence vs domain')
fig.show()


## 9. 📋 consequence pathogenic fraction table


In [27]:
tbl = d.groupby('consequence').agg(pathogenic_fraction=('pathogenic', 'mean'), n=('var_id', 'size'), pathogenic_n=('pathogenic', 'sum')).reset_index().sort_values(['pathogenic_fraction', 'n'], ascending=[False, False])
display(tbl.head(30))


,consequence,pathogenic_fraction,n,pathogenic_n
13,frameshift variant|5 prime UTR variant,1.000000,95,95
20,genic upstream transcript variant,1.000000,5,5
17,frameshift variant|nonsense,1.000000,3,3
5,5 prime UTR variant|frameshift variant,1.000000,2,2
15,frameshift variant|initiator_codon_variant,1.000000,2,2
64,synonymous variant|splice donor variant,1.000000,2,2
9,5 prime UTR variant|nonsense,1.000000,1,1
10,5 prime UTR variant|nonsense|inframe_insertion,1.000000,1,1
14,frameshift variant|5 prime UTR variant|intron ...,1.000000,1,1
19,genic downstream transcript variant,1.000000,1,1


## 10. 📊 rare consequence enrichment


In [28]:
counts = d['consequence'].value_counts(normalize=True)
d['is_rare_consequence'] = d['consequence'].map(counts).fillna(0) < 0.01
tbl = d.groupby('is_rare_consequence').agg(pathogenic_fraction=('pathogenic', 'mean'), n=('var_id', 'size')).reset_index()
display(tbl)
fig = px.bar(tbl, x='is_rare_consequence', y='pathogenic_fraction', hover_data=['n'], title='Rare consequence enrichment for pathogenicity')
fig.show()


,is_rare_consequence,pathogenic_fraction,n
0,False,0.193177,9028
1,True,0.734211,2280


## 11. 🧪 Fisher rare consequence vs pathogenic 📖


In [29]:
tab, odds, p, ci = fisher_bool(d, 'is_rare_consequence', 'pathogenic')
display(tab)
display(pd.Series({'odds_ratio': odds, 'p_value': p, 'or_ci_low': ci[1], 'or_ci_high': ci[2]}).to_frame('value'))


pathogenic,False,True
is_rare_consequence,,
False,7284,1744
True,606,1674


,value
odds_ratio,11.537356
p_value,0.000000
or_ci_low,10.370722
or_ci_high,12.835227


## 12. 📊 entropy consequence distribution 📖


In [30]:
p = d['consequence'].value_counts(normalize=True)
ent = entropy(p.values, base=2)
out = pd.Series({'n_consequences': int(p.shape[0]), 'entropy_bits': float(ent)})
display(out.to_frame('value'))
fig = px.bar(p.head(20).reset_index(), x='consequence', y='proportion', title='Top consequence proportions')
fig.update_xaxes(tickangle=-45)
fig.show()


,value
n_consequences,65.000000
entropy_bits,3.005813
